# Case Study II: Missing values, Outliers, and Simulation Studies

- Name: Luke Clausi
- Student ID: 52447760
- Link to GitHub Repository: https://github.com/lukeclausi/dsci-200-case-study-2

In [ ]:
library(tidyverse)
library(naniar)
library(mice)
library(simputation)
library(infer)
library(matrixStats)
library(cellWise)

Milestone 1 - Missing Vals

(m1t1)

In [ ]:
set.seed(123)
storms_dat <- storms  #q1
dim(storms_dat)       #q2 
glimpse(storms_dat)   #q3 + q4 

(m1t1): 
Loaded dataset to storms_dat. 

The dataset has 19537 observations (rows) and 13 variables (columns). 

Categorical vars: 
- name: character 
- status: factor

Numerical vars:
- year, month, hour, long, lat, category: dbl float
- day, wind, pressure, tropicalstorm_force_diameter, hurricane_force_diameter: integer

Glimpse showed us that category, tropicalstorm_force_diamater, and hurricane_force_diameter all are showing only NA values (in the glimpse, not in the whole set necessarily). 

One advantage of glimpse here is that it is a very quick method to get an overview of the dataset, including variable types and whether missing values are immediately present. A limitation is that is that it does not give good info for discerning patterns of missingness, such as whether missing values occur together in the same observation or within certain groups. 

In [ ]:
# (m1t2)

vis_miss(storms_dat)

gg_miss_var(storms_dat, show_pct = TRUE) 

gg_miss_fct(storms_dat, fct = status)

gg_miss_upset(storms_dat)

ggplot(storms_dat, aes(x = tropicalstorm_force_diameter,
                       y = hurricane_force_diameter)) +
  geom_point()

ggplot(storms_dat, aes(x = tropicalstorm_force_diameter,
                       y = hurricane_force_diameter,
                       colour = status)) +
  geom_miss_point()

(m1t2): Used vis_miss(storms_dat). 
We observe the missing values are concentrated in only 3 variables: category, tropicalstorm_force_diameter, and hurricane_force_diameter. These all contain substantial missingness while other variables appear complete. We also observe the pattern of tropicalstorm_force_diameter and hurricane_force_diameter perfectly sharing their missing/complete observations with each other, while the missingness in category seems to be independent of the missigness in the two connected variables. We also observe that 86.7% of all variable-observations (data cells) are present while 13.3% are missing (overall). 

Used gg_miss_var(storms_dat, show_pct = TRUE). 
We observe similar results and get similar conclusions to vis_miss(). We can now put a percentage to the missingness of each variable: about 48% for tropicalstorm_force_diameter and hurricane_force_diameter, and about 76% for category. This shown percentage-by-variable is an advantage over vis_miss(). 

Used gg_miss_fct(storms_dat, fct = status). 
The plot suggests that the missingness is dependent on storm status. The category variable missigness is fully explained here: this variable is complete for all hurricanes, and fully missing for all non-hurricanes, implying it is only applicable to hurricanes and the data is complete-as-intended regarding category. Meanwhile, we see that the two connected variables with some missingness tropicalstorm_force_diameter and hurricane_force_diameter, which i will referring to as tsfd and hfd for brevity) have some connection to status aswell. Disturbance and other low statuses are complete (or near-complete) in this variable, while other statuses such as subtropical depression and extratropical have middling levels of missigness in these variables. 

Used gg_miss_upset(storms_dat). 
We again verify that missing values often occur together in the same observation rather than independently, as discussed. tsfd and hfd share their missigness (have identical missing and non-missing observations). This graph doesn't tell us anyhting new about category, as we already diagnosed its missingness just being tied to hurricane vs non-hurricane. 

Used ggplot(storms_dat, aes(x = tropicalstorm_force_diameter,
                       y = hurricane_force_diameter)) +
                        geom_point().
ggplot2 excludes rows with missing values in the relevant plotted variables by default. As a result observations with missing tsfd or hfd are omitted from the scatter plot. 

Used ggplot(storms_dat, aes(x = tropicalstorm_force_diameter,
                       y = hurricane_force_diameter,
                       colour = status)) +
                    geom_miss_point().
We coloured points by their status because we diagnosed that the status variable was explanatory for much of the missing data. Hurricanes accounted for the most observations with substantial non-zero values of both tsfd and hfd, while many non hurricane storm types are concentrated near zero or other missing-value regions. This suggests the missingness pattern is related to storm classification and intensity rather than being random. 


Overall, the missingness in this dataset is concentrated in three variables: category, tropicalstorm_force_diameter, and hurricane_force_diameter, while the other variables appear complete. The visualizations suggest that the missingness is not random. In particular, tropicalstorm_force_diameter and hurricane_force_diameter seem to completely share the same missingness pattern and are often missing together in the same observations. By contrast, the missingness in category follows a different pattern. Using gg_miss_fct() with status showed that category is complete for hurricanes and missing for non-hurricanes, which suggests this is structural missingness because storm category is only applicable to hurricanes. The coloured scatter plot with geom_miss_point() also showed that hurricanes account for most observations with substantial nonzero values of both diameter variables, while many non-hurricane storm types are concentrated near zero or missing-value regions. Taken together, these results suggest that missingness in the dataset is strongly related to storm classification and intensity rather than occurring independently or completely at random.

Task 3: 
- run the preprocessing code to create storms_03plus
- identify one variable with structural missingness that should not be imputed
- briefly justify why its missing values are not applicable
- remove that variable from storms_03plus before imputing
- make a scatter plot of tropicalstorm_force_diameter vs pressure
- add a layer showing missing values
- briefly describe the relationship and missingness pattern
- optionally facet by year to see if missingness changes over time
- do mean imputation for tropicalstorm_force_diameter, save as storms_mean
- do regression imputation with pressure only, save as storms_lm_small
- do regression imputation with pressure + wind + lat + long + status, save as storms_lm_large
- complete the mice() code to make storms_mice
- extract one completed dataset as storms_mice1
- bind the four completed datasets together
- use the provided faceted plot to compare methods
- describe how the imputed values differ across methods
- choose two methods and give one limitation of each

In [ ]:
storms_03plus <- storms_dat |>
  filter(year > 2002) |>
  mutate(tropicalstorm_force_diameter = as.numeric(tropicalstorm_force_diameter)) |>
  nabular() |>
  add_label_shadow()

glimpse(storms_03plus)

(m1t3) We identified in t2 that the missigness of category was structural (whether or not the storm is a hurricane, as only the variable is a descriptor of hurricane category and not applicable to other storms). Thus, these are not missing measurements, just undefined values, and we should not impute it here. 

In [ ]:
storms_03plus <- storms_03plus |>  # 15 
  select(-category)

ggplot(storms_03plus, aes(x = pressure, y = tropicalstorm_force_diameter)) +  # 16
  geom_point(alpha = 0.4) +
  geom_miss_point() +
  theme_minimal()

(m1t3) Created a scatter plot with a missing layer, showing tpfd and pressure. We see a negative relationship between pressure and tpfd: as pressure rises, tpfd tends to decrease. Missing values are shown between the main cloud and are not randomly spread. They are concentrated near the top of the pressure range, but still present throughout all pressures. This suggests the missingness may not be completely random, but given this is a crowded scatter-plot without an extremely clear explanation for missingness, we cannot draw any confident conclusions. 

In [ ]:
storms_mean <- storms_03plus |>  # 17 
  mutate(
    tropicalstorm_force_diameter =
      ifelse(
        is.na(tropicalstorm_force_diameter),
        mean(tropicalstorm_force_diameter, na.rm = TRUE),
        tropicalstorm_force_diameter
      )
  )

head(storms_mean)

storms_lm_small <- storms_03plus |>  # 18
  impute_lm(tropicalstorm_force_diameter ~ pressure)


head(storms_lm_small) 

storms_lm_large <- storms_03plus |> #19 
  impute_lm(tropicalstorm_force_diameter ~ pressure + wind + lat + long + status)

head(storms_lm_large)

In [ ]:
glimpse(storms_03plus)

In [ ]:
# (m1t3q20)

storms_03plus_mice <- storms_03plus |>
  select(
    status, wind, pressure,
    tropicalstorm_force_diameter, hurricane_force_diameter
  )

storms_mice <- mice(storms_03plus_mice, m = 3, maxit = 5, seed = 123, printFlag = FALSE)
 
storms_mice1 <- complete(storms_mice, action = 1)


In [ ]:
# (m1t3q21)

bound_models <- bind_rows(
  mean = storms_mean,
  small = storms_lm_small,
  large = storms_lm_large,
  mice = storms_mice1,
  .id = "imp_model"
) |>
  as_tibble()

bound_models |>
  arrange(tropicalstorm_force_diameter_NA) |>
  ggplot(aes(x = pressure,
             y = tropicalstorm_force_diameter,
             colour = tropicalstorm_force_diameter_NA)) +
  geom_point() +
  facet_wrap(~imp_model)

(m1t3q22)

One limitation of mean imputation is that it replaces all missing values with the same constant, which removes natural variability and ignores and linear or other dynamic relationship between pressure and tsfd. In the plot, this shows up as a horizontal turqoise line as the imputed values are set to the mean value. Meanwhile, a limitation of regression imputation with pressure only is that it produces imputed values that must lie exactly across a narrowly fitted line, making the data look overly regular and underestimating variability. It also ignores other storm characteristics that may explain storm size. 

In [ ]:
#saveRDS(storms_mice, "storms_mice.rds")   # saving mice results to restore later if needed, as generation runtime is long

In [ ]:
# Milestone 2: 

storms_complete <- storms |>
     filter(year > 2002) |>
     select(-category) |>
     drop_na()
 
 # re-group storms and change variable type for later imputation
storms_pop <- storms_complete |>
   mutate(type = case_when(
     status == "hurricane" ~ "hurricane",
 
     status %in% c("tropical storm",
                   "tropical depression") ~ "tropical",
 
     status %in% c("subtropical storm",
                   "subtropical depression",
                   "tropical wave",
                   "disturbance",
                   "other low") ~ "subtropical_disturbance",
 
     status == "extratropical" ~ "extratropical"
   ),
     tropicalstorm_force_diameter = as.numeric(tropicalstorm_force_diameter)) 

In [ ]:
tidy(lm(tropicalstorm_force_diameter~pressure + lat + type, storms_pop))

In [ ]:
# (m2t1) Simulating data with MCAR 

## Helper functions

## Generate data
get_sample <- function(dat, n_size){
  storms_sample <- dat |>
    slice_sample(n = n_size, replace = FALSE) |>
    ungroup()
  storms_sample
}

## Generate missingness MCAR
generate_mcar <- function(dat, p) {
  missing_y <- sample(1:nrow(dat), size = round(p * nrow(dat)))
  dat$tropicalstorm_force_diameter[missing_y] <- NA
  dat
}

## Linear regression estimation by least squares (LS)
fit_lm <- function(dat) {
  lm(tropicalstorm_force_diameter ~ pressure + lat + type, data = dat) |>
    tidy() |>
    select(term, estimate, std.error)
}

## Mean imputation, then LS estimation
est_mean_imp <- function(dat) {
  dat |>
    mutate(
      tropicalstorm_force_diameter =
        ifelse(
          is.na(tropicalstorm_force_diameter),
          mean(tropicalstorm_force_diameter, na.rm = TRUE),
          tropicalstorm_force_diameter
        )
    ) |>
    fit_lm()
}

## Regression imputation, then LS estimation
est_lm_imp <- function(dat) {
  dat |>
    impute_lm(tropicalstorm_force_diameter ~ pressure) |>
    fit_lm()
}

## mice imputation, then LS estimation and POOLED
est_mice_imp <- function(dat, m = 3, maxit = 5, seed = 123) {
  imp <- mice(dat, m = m, maxit = maxit, seed = seed, printFlag = FALSE)

  ## pooled across m imputations
  pooled <- pool(with(imp, lm(tropicalstorm_force_diameter ~ pressure + lat + type))) |>
    summary() |>
    select(term, estimate, std.error)

  pooled
}

In [ ]:
set.seed(123)

# take a sample of size 1000
storms_sample <- get_sample(storms_pop, 1000)

# add 30% MCAR missingness to tropicalstorm_force_diameter
storms_mcar <- generate_mcar(storms_sample, 0.30) |>
  nabular() |>
  add_label_shadow()

# scatter plot
ggplot(storms_mcar, aes(y = tropicalstorm_force_diameter,
                        x = pressure)) +
  geom_point(alpha = 0.4) +
  geom_miss_point() +
  theme_minimal()

In [ ]:
#(m2t2)

# mean imputation
storms_mean <- est_mean_imp(storms_mcar)

# regression imputation
storms_lm <- est_lm_imp(storms_mcar)

# mice imputation  
storms_mice <- est_mice_imp(storms_mcar) 

dims(

In [ ]:
cbind(
  fit_lm(storms_sample)[, 1:2],
  complete_estimate   = fit_lm(storms_mcar)$estimate,
  mean_estimate       = storms_mean$estimate,
  regression_estimate = storms_lm$estimate,
  mice_estimate       = storms_mice$estimate
)

In the table above, we can compare the estimated coefficients between our original sample and our attempted methods. 

Mean imputation: This one looks the worst. The original pressure is -6.2, while the one generated here is -4.34 (while all the others are much closer to -6 at least). The lat is also the furthest off (3.13 original vs 1.88 here). Mean imputation must have flattened some relationship that was available in the data, which other methods were able to use to better represent the data, resulting in mean imputation being the worst. 

Complete-case estimate: This one is pretty close to the original pressure and lat, at -5.95 and 3.17. respectively. Other estimates of complete-case are also reasonably close to the original. This makes sense, given we generated missingness as MCAR, so dropping missing rows does not create huge bias in the run, although it does still lose data.

Regression imputation: The regression imputation gets closer than mean imputation, but does not do as well as mice seems to. The pressure is a bit off at -5.7 and the  lat is very far off at 2.1. The type coefficients are noticeably shrunk towards zero. Because we only impute from pressure it is a reasonable theory that regression is losing context from variables other than pressure that mice is able to use to better project values here. 

Mice: This one looks the closest overall, as mentioned, being similar in accuracy to the complete case (which is, by design of using MCAR, going to seem accurate, but may not translate to real-world randomness that does not obey MCAR assignments). The pressure is at -6.2 here while the lat is at 2.9, which is not exact but still closer than other estimators. It appears mice was able to recover the original regression relationships the best.

We note these results come from a single simulated sample so they are not enough to draw general conclusion given the immediate option of doing a more complete simulation study (as we will now). 

In [ ]:
# (m2t3q4) - running that simulation 500 times 

set.seed(123)

datasets <- replicate(
  500,
  {
    simulated_data <- get_sample(storms_pop, 1000)
    generate_mcar(simulated_data, p = 0.30)
  },
  simplify = FALSE
)

complete_coeff <- datasets |>
  map_dfr(fit_lm, .id = "rep")

mean_imp_coeff <- datasets |>
  map_dfr(est_mean_imp, .id = "rep")

lm_imp_coeff <- datasets |>
  map_dfr(est_lm_imp, .id = "rep")

nn <- length(datasets)
mice_imp_coeff <- suppressWarnings(
  map_dfr(seq_along(datasets), function(i) {
    cat("\rRunning dataset", i, "of", nn)
    flush.console()
    est_mice_imp(datasets[[i]])
  }, .id = "rep")
)

cat("\n")

In [ ]:
# (m2t3q5) - creating a summary results set 

coeffs_pressure <- bind_rows(
  complete_coeff  |> mutate(method = "Complete case"),
  mean_imp_coeff  |> mutate(method = "Mean imputation"),
  lm_imp_coeff    |> mutate(method = "Regression imputation"),
  mice_imp_coeff  |> mutate(method = "MICE imputation")
) |>
  filter(term == "pressure")

dim(coeffs_pressure)

(m2t2q5) 

The dimensions of coeffs_pressure are 2000 x 5. 

This makes sense, as there were 500 simulation runs with 4 methods, producing the 2000 rows. Meanwhile, the 5 columns are the regression outputs, which should include the rep, term, estimate, std error, and method label. This produces the 2000 x 5 output. 

In [ ]:
glimpse(coeffs_pressure)

In [ ]:
# (m2t3q6) - histogram seperated by method of regression results

ggplot(coeffs_pressure, aes(x = estimate, fill = method)) +
  geom_histogram(
    bins = 20,
    alpha = 0.5,
    position = "identity"
  ) +
  geom_vline(xintercept = -5.752437, linetype = "dashed") +
  facet_wrap(~ method, scales = "free_y") +
  labs(
    title = "Sampling Distribution of `pressure` Coefficient Estimator",
    x = "Estimated coefficient",
    y = "Count"
  ) +
  theme(legend.position = "none")

In [ ]:
# (m2t3q7) - summary statistics 

pressure_summaries <- coeffs_pressure |>
  group_by(method) |>
  summarize(
    median_beta_hat = median(estimate, na.rm = TRUE),
    median_SE       = median(std.error, na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate_if(is.numeric, round, 3)

pressure_summaries

(m2t3q7) - summary statistic explanation 

We can now compare the median estimated coefficients and median estimated std error across these runs. 

The reference pop value for pressure was about -5.752 (from near the start of milestone 2). 

Complete case: median estimated pressure of -5.733 with a median SE of 0.341. This is very close to our reference, meaning very little bias. The SE being one of the two highest (MICE and Complete case are both around 0.34 for median SE) indicates more uncertainty in the estimates. 

MICE imputation: median estimated pressure of -5.732, which is extremely close to complete case (while still 0.020 away from the reference value). This one also has very little bias. Same median SE as complete case aswell, indicating the same thing about including more uncertainty than the other estimates. 

Mean imputation: -4.022 median estimated pressure value here. This method seems to be by far the least accurate in this case, despite having less  uncertainty than complete case and mice (via lower median SE of 2.8), it seems to include more bias. 

Regression imputation: median estimated pressure of -5.602 is further from reference pressure than both complete case and mice, but much better than mean imputation. This one has the lowest uncertainty (via the smallest median SE of 2.5), but still has more bias than MICE or complete case methods in this case. 

Taking a look at the plot, these descriptions seems pretty visually consistent. The Mice and complete case visualizations are very similar to one another, with a midpoint very close to the vertical dashed line representing the reference value. Meanwhile, the regression imputation is close, but not as close as those two, to centering around the reference, but is noticeably tighter than both mice and complete case curves. Finally, the mean imputation has a centre the furthest from the reference, while having a medium tightness of its curve. 

In [ ]:
# (m3t1q1) - building iris datasets

set.seed(1)

# 1) iris dataset
iris_numeric <- iris |>
  select(-Species)

# 2) Proportional to first flower, different magnitude
base <- as.numeric(iris_numeric[1, ])
factors <- seq(0.5, 2, length.out = 150)
iris_prop <- factors %*% t(base)

# 3) Similar flowers (`dir` keeps dependency among original variables)
dir  <- c(1, 0.8, 1.2, 1.1)
z    <- rnorm(150, sd = 0.02)

iris_similar <- matrix(rep(base, each = 150), nrow = 150) + z %*% t(dir)

# 4) Very different flowers (random, same dimensions)
iris_varied <- matrix(rnorm(150 * 4), ncol = 4)



In [ ]:
# (m3t1q2) - computing pca 

pca_prop    <- prcomp(iris_prop, scale = TRUE)
pca_similar <- prcomp(iris_similar, scale = TRUE)
pca_varied  <- prcomp(iris_varied, scale = TRUE)
pca_iris    <- prcomp(iris_numeric, scale = TRUE)

head(pca_iris$x)

In [ ]:
# (m3t1q3) - computing proportions of explained variance and plotting results

ve_df <- data.frame(
  PC = rep(paste0("PC", 1:4), times = 4),
  dataset = rep(
    c("Proportional rows", "Similar rows", "True data", "Varied rows"),
    each = 4
  ),
  prop_var = c(
    (pca_prop$sdev^2)    / sum(pca_prop$sdev^2),
    (pca_similar$sdev^2) / sum(pca_similar$sdev^2),
    (pca_iris$sdev^2)    / sum(pca_iris$sdev^2),
    (pca_varied$sdev^2)  / sum(pca_varied$sdev^2)
  )
)

ggplot(ve_df, aes(x = PC, y = prop_var, fill = dataset)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     limits = c(0, 1)) +
  labs(
    x = NULL,
    y = "Proportion of variance explained",
    fill = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "top",
    panel.grid.minor = element_blank()
  )

(m3t1q3)  

Besides the proportional rows dataset, the similar rows dataset also has PC1 explain a very large (near 100% in the graph) proportion of the variation. This is because the rows in that dataset are all based on the same template flower with only a small amount of noise introduced. 

The dataset that needs multiple principal components is the varied rows, which has 4 near-quarter blocks in the graph. This is because its variation is explained about proportionally by all 4 PCs. The measurements were generated more independently and with less shared structure so the variation is spread more evenly across the available directions. No single princical component can explain most of its variation. 

In [ ]:
# (m3t2) - loading helpers

source("case-study-2-functions.R")
data(data_dogWalker)

# save data
dogWalker_dat <- data_dogWalker
class(dogWalker_dat)

# (m3t2q4) - reshaping data

show_frame(dogWalker_dat, frame_number = 10)

dims <- dim(dogWalker_dat)
dims

# flatten into a dataframe
dim(dogWalker_dat) <- c(dims[1], prod(dims[2:4]))
dogWalker_dat <- as.data.frame(dogWalker_dat) 

The dimensions are 54 x 144 x 180 x 3. This reflects 54 frames, with each frame of 144 x 180 dimensions, with 3 RGB dimensions. 

(m3t2q5) 

Based on the previous examination of PCA, I expect that much of the information in the frames can be represented with a small number of PCs. The camera is static, so most of the pixels correspond to the background which changes very little frame to frame. This means the rows of the dataset should be highly similar, so most of the variation can be captured with a few dominant components. 

The moving man and his dog can be reasonably thought of as the outliers here. This is because they affect only a relatively small part of the frame while the background remains stable. They behave as cellwise outliers in this case given that they cause only certain pixel values to differ strongly from the background pattern in each frame. Unlike the background, these entities will be moving and affecting the data of the pixels across which they pass (ie, their movement/presence is overlaid on the otherwise static background in that section of the video). 

In [ ]:
# (m3t2q6) - computing PCA 

# The following lines add small variation where scale is zero to avoid division by 0

Xscales <- colSds(as.matrix(dogWalker_dat))
fix <- scale_filler_c(as.matrix(dogWalker_dat), Xscales, dims)

# Start with fixed data
dogWalker_dat <- fix[[1]]

dogWalker_pca <- prcomp(dogWalker_dat, scale = FALSE)

# get the loadings and visualize those of PC1
cloadings <- dogWalker_pca$rotation
cloadings_PC1 <- cloadings[, 1]

## prepare loading for image 
load1_img <- prepLoading(cloadings_PC1, dims[2:4])

# visualization of the loadings of PC1
Pal <- colorRampPalette(c("blue", "white","red"))(100)

image(t(apply(load1_img$loading, 2, rev)),
      zlim = 2 * load1_img$maxab * c(-1, 1),
      col = Pal)

(m3t2q6)

In this image, we can clearly observe a strong shape on the left, with a broad colored region on the right. These are traces of the moving man and his dog from the video. We note the rest of the image is mostly  featureless, indicating the static background in the video. There is a small amount of noise in the background area near the top - given the frame from the video, it would be reasonable to suggest this noise is a result of the tree above the man having a small amount of movement from wind in the video, but not nearly as much pixel change as the man and the dog in the primary bottom section of the image. 

We note the image's red/blue pixels both indicate a difference (a change) in those pixels thoughout the video, while the paler areas indicate a lack of change in those pixel assignments throughout the video. Overall, this image largely corroborates our earlier hypothesis, with the addition of a small amount of movement in the tree above the man. We primarily should note that, due to the man and dog being clearly flagged in this result, classical PCA will struggle to produce the original background, specifically in the area which these outliers passed through in the frame. 

In [ ]:
# (m3t2q7) - robust PCA

locScaleX <- estLocScale(as.matrix(dogWalker_dat), precScale = 1e-12)

# add minimum variation to the data if scale is close to 0 and re-estimate scale
fix <- scale_filler_r(as.matrix(dogWalker_dat), locScaleX$scale, dims)

data_filled <- fix[[1]]
scale_fix <- fix[[2]]

# wrap `data_filled` using estimate of location and reviewed scale
Xw <- wrap(data_filled, locScaleX$loc, scale_fix)$Xw

# center wrapped data
XwC <- sweep(Xw, 2, colMeans(Xw))

# robust PCA: compute PCA on centered-wrapped data
dogWalker_rpca <- prcomp(XwC, center = FALSE, scale. = FALSE)

# get the loadings of the robust PC1
rloadings <- dogWalker_rpca$rotation
rloadings_PC1 <- rloadings[, 1]

# prepare loadings of PC1 for image
rload1_img <- prepLoading(rloadings_PC1, dims[2:4])

# visualization of the loadings of robust PC1
image(t(apply(rload1_img$loading, 2, rev)),
      zlim = 2 * rload1_img$maxab * c(-1, 1),
      col = Pal)

(m3t2q7) 

Compared with the classical PCA loading image, the robust PCA loading image shows much weaker traces of the moving man and dog. The frame is overall paler and less dominated by the path of motion, although some weaker variation remains near the tree branches at the top. This suggests that robust PCA reduces the influence of the moving outlier pixels on the first principal component and produces a cleaner dominant background pattern than classical PCA.

In [ ]:
# (m3t3q8) - isolating moving objects, computing classical PCA residuals

# Compute the fitted frames and residuals:
aveX <- colMeans(dogWalker_dat)
Xresidual <- compute_residuals(dogWalker_dat, cloadings, aveX, k = 3)

# residuals as images
locs <- colMeans(Xresidual)
scales <- colSds(Xresidual)
show_frame_residual(10, dogWalker_dat, Xresidual, locs, scales, 20)

(m3t3q8)

This is the classical PCA's residual of a frame in the video, side by side with the original frame. It is indicating what remains after subtracting the PCA reconstruction from the original frame. It is meant to show us what about the reconstruction is notably different from the original frame of the same time in the video. In the residual, we observe the torso, head, and legs of a man. A little bit of the dog's leg are showing up in the residual, but without the context of the original frame, it is not possible to identify those pixels as a dog's outline. A better result here would show the man and the dog with clear outlines and full pixel sections across their bodies, with minimal noise in the rest of the frame. 

In [ ]:
# (m3t3q8) - robust PCA residuals for comparison 

aveXw <- colMeans(XwC)
Xrresidual <- compute_residuals(XwC, rloadings, aveXw, k = 3)

locScale_res <- estLocScale(Xrresidual)
show_frame_residual(10, dogWalker_dat, Xrresidual, locScale_res$loc, locScale_res$scale, 20)

(m3t3q9)

In the robust PCA's residual, the moving man and dog are (unlike the classical) not visibly highlighted at all. Under the interpretation suggested by instructions, this means the robust PCA has retained the moving objects (man and his dog) instead of isolating them in the residuals. Even with the same values for the cutoff in either residual, this still remains true - the man not appearing in the residual of the robust PCA implies he is in both the original image and the reconstructed image. This suggests the frames that robust PCA reconstructed in this case did not isolate the moving objects. 

(m3t3q10)

Trying varius cutoffs in the residual-generation indicated the following: 

The classical PCA residual contains a blurry/spotty man. By definition, this incomplete outline of the man in the residual means that the PCA generated the background accurately to the video's frame, but did not entirely remove the man - likely leaving some ghost-like traces of him and the dog, as if they were to be entirely gone in the reconstruction, the classical PCA residual would show both of them in their entirety with a much clearer outline. 

In contrast, the robust PCA residual does not contain a substantial outline of either the man or the dog. Since the man is clearly visible in the original frame but does not appear in the residual image, the displayed result suggests that the robust PCA reconstruction retained the moving objects instead of isolating them. 

Therefore, based on the residual images produced by the provided code, classical PCA appears more effective at visibly isolating the moving objects, although it does so imperfectly, while robust PCA appears to retain them in the reconstructed frames rather than separating them into the residuals. 

We can connect back to Q6 and Q7 here aswell - the loading images of both PCA cases. The classical PCA loading image in Q6 showed clear traces of the moving man and dog - which indicated that motion had leaked into the first principal component. In Q8, the classicial residual image still showed a ghost-like partial outline of the man, with the remainder of that ghost being the reconstruction, so classical PCA did not isolate the motion cleanly: some of it leaked into the PCA component, while some remained in the residual. 

Meanwhile, in Q7, the robust PCA loading image showed much weaker traces of moving objects, suggesting less contamination of the first principal component. However, in Q9, the robust residual image did not visibly highlight the moving man or dog. As suggested, this means the moving objects are retained in the reconstruction instead of being isolated in the residuals. Therefore, classical PCA appears worse at keeping motion out of the principal component, but better at visibly flagging motion in the residual image, while robust PCA appears better at reducing contamination of the first component but worse at isolating the moving objects in the displayed results. 

This mismatch could potentially be explained by acknowledging that the loading images were done based on one primary component - so even though the robust PCA loading imaged seemed to successfully mute the motion, that was only in the first primary component - and other primary components managed to encode that motion accurately (more accurately than all of the classical PCA case). This would allow the robust case to include moving objects in its reconstruction. Our classical case is still consistent with this - the first primary component was polluted by motion outliers, which still resulted in a ghostly inclusion of those moving objects in the reconstruction. 